In [24]:
# Import library
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer 
import pandas as pd
from transformers import pipeline
import torch
from tqdm import tqdm

In [ ]:
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU")

#### Load the Datasets

In [23]:
# Read the csv file
teaser = pd.read_csv("./data/teaser_comments.csv")
debut = pd.read_csv("./data/debut_comments.csv")


# debut_test = debut[:10000]
# teaser_test = teaser[:10000]

In [8]:
teaser.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32705 entries, 0 to 32704
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   group        32705 non-null  object
 1   video_id     32705 non-null  object
 2   publishedAt  32705 non-null  object
 3   author       32705 non-null  object
 4   text         32705 non-null  object
 5   likeCount    32705 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 1.5+ MB


In [21]:
analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    scores = analyzer.polarity_scores(text)
    #print("Text", text, "Score", scores)

    return scores["neg"], scores["neu"], scores["pos"], scores["compound"]

def get_vader_label(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

#### Run Vader Sentiment Analysis on Teaser Comments:

In [10]:
vader_sentiments = []
vader_compounds = []

for text in teaser["text"]:
    vader_scores = get_vader_scores(text)
    vader_label = get_vader_label(vader_scores[3])
    vader_sentiments.append(vader_label)
    vader_compounds.append(vader_scores[3])
    
teaser["vader_sentiment"] = vader_sentiments
teaser["vader_compound"] = vader_compounds

teaser.head(20)

,group,video_id,publishedAt,author,text,likeCount,vader_sentiment,vader_compound
0,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:59:48Z,@MeelyBieber,Le sserafim motomamiiiiis,0,neutral,0.0000
1,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:58:54Z,@allygarska,<b>INDUSTRY TAKEOVER</b>,1,neutral,0.0000
2,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:58:04Z,@naswacitrarevalina6716,slayyy,1,neutral,0.0000
3,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:56:06Z,@venvop9674,we gonna wait another day for teaser 2 t-t I&#...,3,negative,-0.4359
4,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:55:45Z,@mess_mess_mess,わしはルセラフィムを応援し続ける覚悟があります,4,neutral,0.0000
5,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:54:32Z,@zhamiradaffaandytra8185,pusing cakep-cakep mereka T__T,2,neutral,0.0000
6,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:52:11Z,@vwiinterrb,KECEEE KECEEE ABIS!!,2,neutral,0.0000
7,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:51:53Z,@vwiinterrb,keceee bgt woi,2,neutral,0.0000
8,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:50:55Z,@soulmateofseulgi,"<a href=""https://www.youtube.com/watch?v=A4twV...",0,neutral,0.0000
9,LE SSERAFIM,A4twVWwOueU,2022-04-28T20:49:47Z,@Problematicbaka,Bighits gonna surprise us again ✨,0,positive,0.5267


In [11]:
print(teaser["vader_sentiment"].value_counts())
print(teaser["vader_compound"].mean())

vader_sentiment
neutral     17675
positive    11477
negative     3553
Name: count, dtype: int64
0.16621848952759516


#### Run the Vader Analysis on Debut Comments:

In [12]:
vader_sentiments = []
vader_compounds = []

for text in debut["text"]:
    vader_scores = get_vader_scores(text)
    vader_label = get_vader_label(vader_scores[3])
    vader_sentiments.append(vader_label)
    vader_compounds.append(vader_scores[3])
    
debut["vader_sentiment"] = vader_sentiments
debut["vader_compound"] = vader_compounds

debut.head(20)

,group,video_id,publishedAt,author,text,likeCount,vader_sentiment,vader_compound
0,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:58:41Z,@bichinabichina4452,mid,0,neutral,0.0000
1,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:41Z,@кирусядеточкина,"THEY&#39;RE SOOO SWEET I REALLY LOVE IT, GUYS ...",0,positive,0.9805
2,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:40Z,@Mama_Aqila01,Vibes nya prince bgt ga sih? Para prince yg la...,0,positive,0.7096
3,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:24Z,@leeknooooow,이 옵션을 선택하면 이 옵션을 선택할 수 있습니다♥️😍😅,1,positive,0.8658
4,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:08Z,@loove143_,0:14 &quot;representing KOZ&quot; I can&#39;t ...,146,neutral,-0.0258
5,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:05Z,@rrraissy,these guys are crazy!!,2,negative,-0.4559
6,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:55:49Z,@solhwa4334,americacore,0,neutral,0.0000
7,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:54:42Z,@miorionz,QUE BUEN GRUPO VOY A SEGUIRLOS,0,neutral,0.0000
8,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:54:01Z,@SVGARTiTZ,LETS GOOO,0,neutral,0.0000
9,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:53:13Z,@paigemc2107,I&#39;m into it!! This is a vibe!! 😆,1,positive,0.5673


In [13]:
print(debut["vader_sentiment"].value_counts())
print(debut["vader_compound"].mean())

vader_sentiment
neutral     82038
positive    75191
negative    17349
Name: count, dtype: int64
0.23983822703891672


### Roberta Sentimental Analysis:

In [25]:
device = 0 if torch.cuda.is_available() else -1

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
    truncation=True,
    max_length=512
)

def roberta_sentiment(texts, batch_size=32):
    texts = ["" if not isinstance(t, str) else t for t in texts]

    labels = []
    scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]

        results = sentiment_model(
            batch,
            truncation=True,
            max_length=512
        )

        labels.extend([r["label"] for r in results])
        scores.extend([r["score"] for r in results])

    return labels, scores

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


#### Run the Roberta Sentimental Analysis on Teaser:

In [26]:
BATCH = 64

labels, scores = roberta_sentiment(teaser["text"].tolist(), batch_size=BATCH)

teaser["roberta_sentiment"] = labels
teaser["roberta_confidence"] = scores

teaser.head(20)

  1%|          | 3/512 [00:08<23:51,  2.81s/it]


KeyboardInterrupt: 

In [16]:
print(teaser["roberta_sentiment"].value_counts())
print(teaser["roberta_confidence"].median())

roberta_sentiment
positive    20586
neutral     10945
negative     1174
Name: count, dtype: int64
0.8349221348762512


#### Run the Roberta Sentimental Analysis on Debut:

In [27]:
BATCH = 64

labels, scores = roberta_sentiment(debut["text"].tolist(), batch_size=BATCH)

debut["roberta_sentiment"] = labels
debut["roberta_confidence"] = scores

debut.head(20)

100%|██████████| 2728/2728 [1:41:08<00:00,  2.22s/it]


,group,video_id,publishedAt,author,text,likeCount,roberta_sentiment,roberta_confidence
0,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:58:41Z,@bichinabichina4452,mid,0,neutral,0.559092
1,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:41Z,@кирусядеточкина,"THEY&#39;RE SOOO SWEET I REALLY LOVE IT, GUYS ...",0,positive,0.980104
2,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:40Z,@Mama_Aqila01,Vibes nya prince bgt ga sih? Para prince yg la...,0,positive,0.919478
3,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:24Z,@leeknooooow,이 옵션을 선택하면 이 옵션을 선택할 수 있습니다♥️😍😅,1,positive,0.815676
4,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:08Z,@loove143_,0:14 &quot;representing KOZ&quot; I can&#39;t ...,146,positive,0.882579
5,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:57:05Z,@rrraissy,these guys are crazy!!,2,negative,0.691363
6,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:55:49Z,@solhwa4334,americacore,0,neutral,0.773116
7,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:54:42Z,@miorionz,QUE BUEN GRUPO VOY A SEGUIRLOS,0,neutral,0.744257
8,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:54:01Z,@SVGARTiTZ,LETS GOOO,0,positive,0.762208
9,BOYNEXTDOOR,jizAb-SLvtM,2023-05-25T20:53:13Z,@paigemc2107,I&#39;m into it!! This is a vibe!! 😆,1,positive,0.986661


In [29]:
print(debut["roberta_sentiment"].value_counts())
print(debut["roberta_confidence"].median())

roberta_sentiment
positive    106368
neutral      56859
negative     11351
Name: count, dtype: int64
0.8494698405265808


### Store the Dataset with Sentimental Anlaysis into a new CSV file

In [28]:
# read the teaser sentimental analysis into a new csv
#teaser.to_csv('./data/teaser_sentimental_analysis.csv', index = False)

# read the debut sentimental analysis into a new csv
debut.to_csv('./data/debut_sentimental_analysis_roberta.csv', index = False)